# Cleaning Method for AEMO Data

This notebook steps through the cleaning methods needed to transform the raw AEMO data into a useable dataset to be imported into Microsoft Power BI. It is informed by findings from the 01_exploration notebook. The stable functions live in src/transform.py

It also calculates a number of features that will be used in the analysis

### Cleaning Decisions Summary
- renaming columns to more user friendly names
- explicitly casting data types
- standardising timezone to AEST no DST as per AEMO


                       - canonical column names + types
                       - canonical timezone (NEM time, AEST no DST)
                       - duplicates: drop, keeping last
                       - missing intervals: flag, don't fill
                       - spike threshold: $300/MWh (justify)
                       - features to engineer (list)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path().resolve().parent / "src"))

In [ ]:
import pandas as pd
import matplotlib as plt
from config import PROCESSED_DIR
import seaborn as sns
import numpy as np

In [ ]:
historical = pd.read_parquet(PROCESSED_DIR / "historical_price_demand.parquet")

The column names as provided by AEMO are not intuitive. They've been replaced here, and units of measurement are included in the header

In [ ]:
from transform import rename_cols

historical = rename_cols(historical)
print(historical)


From the exploration step, the "PERIODTYPE" column contains only one value, "Trade." It is not useful information and is removed. 

In [ ]:
from transform import drop_redundant_cols
print(historical["PERIODTYPE"].unique())
historical = drop_redundant_cols(historical)
print(historical.columns)

This step explicity converts data types so that numerical columns are floats, dates are datetime and the State is category

In [ ]:
from transform import cast_types
print(historical.dtypes)
historical = cast_types(historical)
print(f"\n{historical.dtypes}")

AEMO Time is AEST without Daylight Saving. Pandas imports as US datetime, so this needs to be converted

In [ ]:
from transform import standardise_timezone
print(historical["Settlement Date"].dtype)
historical = standardise_timezone(historical)
print(historical["Settlement Date"].dtype)

We need to check whether there are missing dates in our time series, and decide how to handle them. As the data contains duplicate time stamps per State, we need to loop through each state

In [ ]:
from transform import flag_missing_time_intervals
historical = flag_missing_time_intervals(historical)

To aid analysis, a number of features can be engineered from the timestamp in "Date" column. 
hour of day, day of week, day name, month, season, year, weekend flag, public-holiday flag (optional)

In [ ]:
from transform import date_features
print(historical.head(2))
historical = date_features(historical)

print(f"\n******************************************")
print(f"\n{historical.head(5)}")
print(f"\n{historical.tail(5)}")

In [ ]:
from transform import spike
historical = spike(historical)
print(historical["Is Spike"].value_counts())

In [ ]:
# from transform import add_rolling_features
# historical = add_rolling_features(historical, col="Price ($/MWh)", aggs=["mean", "std", "max", "min"], windows={"24h": "1D", "7d": "7D","30d":"30D"})
# print(historical)
NSW = historical[historical["State"]== "NSW1"]
NSW = NSW.set_index(["Settlement Date"])
NSW = NSW.sort_index()

In [ ]:
NSW["Rolling 24hr"]= NSW.rolling("1D")["Price ($/MWh)"].mean()
NSW["Rolling 1hr"]= NSW.rolling("1h")["Price ($/MWh)"].mean()
print(NSW['Rolling 1hr'])

In [ ]:
def add_rolling_features(df: pd.DataFrame, col: str, windows: list[str], aggs: list[str] = ("mean",),) -> pd.DataFrame:
    """calculates a range of rolling statistics over different time periods"""
    df = df.sort_values(["State", "Settlement Date"]).reset_index(drop=True)
    grouped = df.groupby("State", observed=True)

    for window in windows:
        min_period = pd.Timedelta(window) // pd.Timedelta("5min")
        for agg in aggs:
            rolled = (
                grouped.rolling(window, on="Settlement Date", min_periods=min_period)[col]
                .agg(agg)
                .reset_index(level=0, drop=True)
                .reset_index(drop=True)
            )
            df[f"Rolling {window} {agg.title()} {col}"] = rolled
    return df

df = add_rolling_features(historical, "Price ($/MWh)", ["1h","1D", "7D", "30D"], aggs=["mean", "std", "min", "max"])
df.to_csv("test.csv")

In [ ]:
nsw = df[df["State"]=="NSW1"]
# nsw.set_index("Settlement Date")
print(nsw)

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure()            
fig, ax = plt.subplots(figsize=(10, 4))

# ax.plot(nsw["Settlement Date"], nsw["Price ($/MWh)"])
# ax.plot(nsw["Settlement Date"], nsw["Rolling 1h Mean Price ($/MWh)"])
ax.plot(nsw["Settlement Date"], nsw["Rolling 1D Mean Price ($/MWh)"])
ax.plot(nsw["Settlement Date"], nsw["Rolling 7D Mean Price ($/MWh)"])
ax.plot(nsw["Settlement Date"], nsw["Rolling 30D Mean Price ($/MWh)"])